# Generation of the different graphs view

In [32]:
# Auto reload of the modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import os
print(os.environ.get("PYTHONPATH"))
PROJECT_ROOT = os.environ.get("PYTHONPATH")

C:/Users/BISITE/Desktop/GNN_CoPiPred


In [31]:
from pathlib import Path

import torch
from torch_geometric.data import Data

In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PROJECT_ROOT = Path.cwd().resolve().parents[2]
print(f"Project root directory: {PROJECT_ROOT}")

Using device: cuda
Project root directory: C:\Users\BISITE\Desktop\GNN_CoPiPred


In [34]:
embeddings_dir = [str(PROJECT_ROOT / f'data\\epitope3d\\embeddings\\ESM2'), str(PROJECT_ROOT / f'data\\epitope3d\\embeddings\\ESMIF1')]

pdb_dir = str(PROJECT_ROOT / 'data\\epitope3d\\pdb_structure')

## Generate roh_1 (spatial 1) with a spatial threshold of 10 Angstroms

In [35]:
import joblib

def ensure_num_nodes(graph_list):
    """
    Explicitly set num_nodes attribute on all graphs to avoid PyG warning.
    """
    for data in graph_list:
        if not hasattr(data, 'num_nodes') or data.num_nodes is None:
            data.num_nodes = data.node_attrs.shape[0]
    return graph_list

original_test = joblib.load(str(PROJECT_ROOT / 'data\\epitope3d\\epitope3d_test_data.pkl'))
original_train = joblib.load(str(PROJECT_ROOT / 'data\\epitope3d\\epitope3d_train_data.pkl'))

# Ensure num_nodes is set on original graphs
original_test = ensure_num_nodes(original_test)
original_train = ensure_num_nodes(original_train)

Shuffling and creating a validation dataset

In [36]:
import random
random.shuffle(original_train)
random.shuffle(original_test)

epitope3d_val = original_train[:int(0.2 * len(original_train))]
epitope3d_train = original_train[int(0.2 * len(original_train)):]

# Ensure num_nodes is set after splitting
epitope3d_val = ensure_num_nodes(epitope3d_val)
epitope3d_train = ensure_num_nodes(epitope3d_train)

In [37]:
from scipy.spatial.distance import cdist
import numpy as np

def add_edge_distances(graph_list):
    """
    Adding an attribute 'edge_attr' that contains the euclidian distance between every connected residues
    """
    updated_graphs = []
    
    for data in graph_list:
        # Explicitly set num_nodes to avoid PyG warning
        if not hasattr(data, 'num_nodes') or data.num_nodes is None:
            data.num_nodes = data.node_attrs.shape[0]
        
        # Extracting data
        coords = data.coords.cpu().numpy() if data.coords.is_cuda else data.coords.numpy()
        edge_index = data.edge_index.cpu().numpy() if data.edge_index.is_cuda else data.edge_index.numpy()
        
        # Computing every distance
        distance_matrix = cdist(coords, coords, metric='euclidean')
        
        # Exctracting the distance calculated
        edge_distances = distance_matrix[edge_index[0], edge_index[1]]
        
        # Adding edge_attr to the data container
        data.edge_attr = torch.tensor(edge_distances, dtype=torch.float).unsqueeze(1)
        
        updated_graphs.append(data)
    
    return updated_graphs

original_test = add_edge_distances(original_test)
epitope3d_train = add_edge_distances(epitope3d_train)
epitope3d_val = add_edge_distances(epitope3d_val)

In [38]:
epitope3d_test = original_test

In [39]:
torch.save(epitope3d_test, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_1\\epitope3d_TEST.pt'))
torch.save(epitope3d_val, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_1\\epitope3d_VAL.pt'))
torch.save(epitope3d_train, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_1\\epitope3d_TRAIN.pt'))

In [40]:
len(epitope3d_test), len(epitope3d_val), len(epitope3d_train)

(45, 39, 160)

## Generate roh_2 (spatial 2) with KNN approach

In [41]:
for k, v in epitope3d_train[0].items():
    print(k, type(v), v.shape if hasattr(v, 'shape') else None)

edge_index <class 'torch.Tensor'> torch.Size([2, 6344])
y <class 'torch.Tensor'> torch.Size([376])
coords <class 'torch.Tensor'> torch.Size([376, 3])
node_id <class 'list'> None
node_attrs <class 'torch.Tensor'> torch.Size([376, 1792])
num_nodes <class 'int'> None
name <class 'str'> None
train_mask <class 'torch.Tensor'> torch.Size([376])
rsa <class 'list'> None
edge_attr <class 'torch.Tensor'> torch.Size([6344, 1])


In [42]:
from KNN_view import apply_knn_to_graphs

epitope3d_knn_train = apply_knn_to_graphs(epitope3d_train, k=15)
epitope3d_knn_test = apply_knn_to_graphs(epitope3d_test, k=15)
epitope3d_knn_val = apply_knn_to_graphs(epitope3d_val, k=15)

GPU détecté: NVIDIA RTX A4000
CUDA disponible: True
GPU détecté: NVIDIA RTX A4000
CUDA disponible: True
GPU détecté: NVIDIA RTX A4000
CUDA disponible: True


In [43]:
torch.save(epitope3d_knn_train, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_2\\epitope3d_TRAIN.pt'))
torch.save(epitope3d_knn_test, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_2\\epitope3d_TEST.pt'))
torch.save(epitope3d_knn_val, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\spatial_2\\epitope3d_VAL.pt'))

## Generate roh_3 (sequential 3)

In [44]:
epitope3d_train[0]

Data(edge_index=[2, 6344], y=[376], coords=[376, 3], node_id=[376], node_attrs=[376, 1792], num_nodes=376, name='6J5C', train_mask=[376], rsa=[376], edge_attr=[6344, 1])

In [45]:
from src.feature_generation.create_views.sequential_view import create_sequential_graph_k_rank
epitope3d_seq1_train = create_sequential_graph_k_rank(epitope3d_train, k=1)
epitope3d_seq1_test = create_sequential_graph_k_rank(epitope3d_test, k=1)
epitope3d_seq1_val = create_sequential_graph_k_rank(epitope3d_val, k=1)

Processing graph: 6J5C with 376 nodes

Processing graph: 6O22 with 164 nodes

Processing graph: 1K52 with 72 nodes

Processing graph: 4AYE with 243 nodes

Processing graph: 1W81 with 364 nodes

Processing graph: 4HKJ with 285 nodes

Processing graph: 6M17 with 183 nodes

Processing graph: 2Q21 with 171 nodes

Processing graph: 1CSG with 120 nodes

Processing graph: 2X7R with 51 nodes

Processing graph: 4CMM with 117 nodes

Processing graph: 1HIJ with 129 nodes

Processing graph: 1UB4 with 75 nodes

Processing graph: 3P2T with 194 nodes

Processing graph: 3J27 with 495 nodes

Processing graph: 1QYM with 223 nodes

Processing graph: 1XKG with 298 nodes

Processing graph: 3TGQ with 336 nodes

Processing graph: 1R4P with 350 nodes

Processing graph: 2LMS with 165 nodes

Processing graph: 3MWR with 252 nodes

Processing graph: 3HH2 with 218 nodes

Processing graph: 3FCU with 457 nodes

Processing graph: 3LB6 with 109 nodes

Processing graph: 1DB2 with 377 nodes

Processing graph: 4K71 with 

In [46]:
# Print the neigbhboring nodes of the first graph in the training set for both views
print("sequential view neighbors (first graph):")
print(epitope3d_seq1_train[0])
print(epitope3d_seq1_train[0].edge_index)

sequential view neighbors (first graph):
Data(edge_index=[2, 750], edge_attr=[750], y=[376], node_attrs=[376, 1792], coords=[376, 3], node_id=[376], num_nodes=376, name='6J5C', train_mask=[376], rsa=[376])
tensor([[  0,   1,   1,  ..., 374, 374, 375],
        [  1,   0,   2,  ..., 373, 375, 374]])


In [47]:
torch.save(epitope3d_seq1_train, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_1\\epitope3d_TRAIN.pt'))
torch.save(epitope3d_seq1_test, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_1\\epitope3d_TEST.pt'))
torch.save(epitope3d_seq1_val, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_1\\epitope3d_VAL.pt'))

## Generate roh_4 (sequential 2)

In [48]:
from src.feature_generation.create_views.sequential_view import create_sequential_graph_k_rank

epitope3d_seq2_train = create_sequential_graph_k_rank(epitope3d_train, k=2)
epitope3d_seq2_test = create_sequential_graph_k_rank(epitope3d_test, k=2)
epitope3d_seq2_val = create_sequential_graph_k_rank(epitope3d_val, k=2)

Processing graph: 6J5C with 376 nodes

Processing graph: 6O22 with 164 nodes

Processing graph: 1K52 with 72 nodes

Processing graph: 4AYE with 243 nodes

Processing graph: 1W81 with 364 nodes

Processing graph: 4HKJ with 285 nodes

Processing graph: 6M17 with 183 nodes

Processing graph: 2Q21 with 171 nodes

Processing graph: 1CSG with 120 nodes

Processing graph: 2X7R with 51 nodes

Processing graph: 4CMM with 117 nodes

Processing graph: 1HIJ with 129 nodes

Processing graph: 1UB4 with 75 nodes

Processing graph: 3P2T with 194 nodes

Processing graph: 3J27 with 495 nodes

Processing graph: 1QYM with 223 nodes

Processing graph: 1XKG with 298 nodes

Processing graph: 3TGQ with 336 nodes

Processing graph: 1R4P with 350 nodes

Processing graph: 2LMS with 165 nodes

Processing graph: 3MWR with 252 nodes

Processing graph: 3HH2 with 218 nodes

Processing graph: 3FCU with 457 nodes

Processing graph: 3LB6 with 109 nodes

Processing graph: 1DB2 with 377 nodes

Processing graph: 4K71 with 

In [49]:
# Print the neigbhboring nodes of the first graph in the training set for both views
print("Spatial view neighbors (first graph):")
print(epitope3d_seq2_train[0])

Spatial view neighbors (first graph):
Data(edge_index=[2, 748], edge_attr=[748], y=[376], node_attrs=[376, 1792], coords=[376, 3], node_id=[376], num_nodes=376, name='6J5C', train_mask=[376], rsa=[376])


In [50]:
torch.save(epitope3d_seq2_train, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_2\\epitope3d_TRAIN.pt'))
torch.save(epitope3d_seq2_test, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_2\\epitope3d_TEST.pt'))
torch.save(epitope3d_seq2_val, str(PROJECT_ROOT / 'data\\epitope3d\\graph_list\\sequential_2\\epitope3d_VAL.pt'))

check

In [51]:
epitope3d_test[0].edge_index.shape, epitope3d_knn_test[0].edge_index.shape, epitope3d_seq1_test[0].edge_index.shape, epitope3d_seq2_test[0].edge_index.shape

(torch.Size([2, 6312]),
 torch.Size([2, 5610]),
 torch.Size([2, 746]),
 torch.Size([2, 744]))

In [52]:
len(epitope3d_train), len(epitope3d_seq2_train), len(epitope3d_knn_train), len(epitope3d_train)

(160, 160, 160, 160)